# CallShield: Real-Time Spam and Fraud Call Detection
### Complete End-to-End Machine Learning Pipeline
**Binary Classification: SPAM vs NOT SPAM**

---
### Final Workflow Architecture:
1. **Raw Dataset Loading**
2. **Data Cleaning & Quality Audit**
3. **Label Validation (`0,0`, `1,0`, `1,1`)**
4. **Create Derived Binary Target: `spam_target`**
5. **EDA: Primary Spam vs Not Spam Distribution**
6. **Secondary Supporting Analysis: Spam Composition (Non-Fraud vs Fraud)**
7. **EDA: Transcript Length & Word Clouds**
8. **Text Preprocessing & NLP Lemmatization**
9. **Train / Test Split (80/20 Stratified)**
10. **Feature Extraction (TF-IDF + Chi2 Feature Selection) — Target Leakage Prevention**
11. **Train 5 ML Models (Logistic Regression, Naive Bayes, SVM, KNN, Random Forest)**
12. **Performance Metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC)**
13. **Binary 2x2 Confusion Matrices**
14. **Hyperparameter Tuning (GridSearchCV)**
15. **Final Spam vs Not Spam Inference Demonstration**


In [ ]:
# Install and import required libraries
!pip install -q scikit-learn pandas numpy matplotlib seaborn wordcloud nltk

import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


## 1. Raw Dataset Loading & Inspection


In [ ]:
# Locate dataset file locally or in Google Colab environment
dataset_candidates = [
    "dataset.csv",
    "/content/dataset.csv",
    "/content/dataset (1).csv",
    "attached_assets/dataset_1790343886829.csv",
    "attached_assets/dataset.csv"
]
dataset_path = next((p for p in dataset_candidates if os.path.exists(p)), "dataset.csv")
print(f"Loading dataset from: {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 2. Data Cleaning & Label Validation


In [ ]:
# Inspect data schema, missing values, and raw duplicates
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print(f"\nDuplicate rows count: {df.duplicated().sum()}")


### Label Validation
The dataset contains two raw labels: `spam` (or `is_spam`) and `fraud` (or `is_confidential`).

| `spam` | `fraud` | Interpretation | Action in Pipeline |
| :---: | :---: | :--- | :--- |
| **0** | **0** | **Legitimate / Not Spam** | Retained as **Not Spam** |
| **1** | **0** | **Spam, but not necessarily Fraud** | Retained as **Spam** (Do NOT delete or merge) |
| **1** | **1** | **Spam + Fraud** | Retained as **Spam** |

**Guiding Principles:**
- Do NOT remove any of the three combinations. They contain valuable real-world call patterns.
- Do NOT treat `1,0` as Not Spam.
- Do NOT merge, delete, or re-code `1,0` into `0,0`.
- Do not artificially modify the dataset to balance classes or fabricate metrics.


In [ ]:
# Detect column names dynamically
spam_col = "spam" if "spam" in df.columns else "is_spam"
fraud_col = "fraud" if "fraud" in df.columns else "is_confidential"
print(f"Spam column: '{spam_col}', Fraud column: '{fraud_col}'")

# Validate label combination counts
combo_counts = df.groupby([spam_col, fraud_col]).size()
print("\nLabel combination distribution in raw dataset:")
print(combo_counts)

c_0_0 = int(((df[spam_col] == 0) & (df[fraud_col] == 0)).sum())
c_1_0 = int(((df[spam_col] == 1) & (df[fraud_col] == 0)).sum())
c_1_1 = int(((df[spam_col] == 1) & (df[fraud_col] == 1)).sum())
c_0_1 = int(((df[spam_col] == 0) & (df[fraud_col] == 1)).sum())

print(f"\nVerified Label Breakdown:")
print(f"  • (0,0) Legitimate / Not Spam:           {c_0_0} calls")
print(f"  • (1,0) Spam, but not necessarily Fraud: {c_1_0} calls")
print(f"  • (1,1) Spam + Fraud:                    {c_1_1} calls")
if c_0_1 > 0:
    print(f"  • (0,1) Legitimate with security/conf.:  {c_0_1} calls")


## 3. Create Derived Target: `spam_target`

For the primary **Spam vs Not Spam** analysis and ML models:
```text
if spam == 0:
    spam_target = 0   # Not Spam / Legitimate

if spam == 1:
    spam_target = 1   # Spam
```
Therefore:
- `0,0` → **Not Spam** (`spam_target = 0`)
- `1,0` → **Spam** (`spam_target = 1`)
- `1,1` → **Spam** (`spam_target = 1`)

The total Spam count is `number of (1,0) + number of (1,1)`.
The total Not Spam count is `number of (0,0)`.


In [ ]:
# Create spam_target according to the explicit specification
df["spam_target"] = df[spam_col].apply(lambda s: 1 if s == 1 else 0)

not_spam_total = int((df["spam_target"] == 0).sum())
spam_total = int((df["spam_target"] == 1).sum())

print("Target variable 'spam_target' created successfully:")
print(f"  Total Not Spam (0): {not_spam_total} calls")
print(f"  Total Spam (1):     {spam_total} calls (sum of 1,0 [{c_1_0}] + 1,1 [{c_1_1}])")
print(f"  Total records:      {len(df)}")


## 4. EDA — Primary Binary Distribution: Spam vs Not Spam
Visualizing the primary binary classification task (`Spam` vs `Not Spam`), showing exactly two classes without confusing three-class artifacts.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Bar Chart: Spam vs Not Spam
palette = ["#16a34a", "#dc2626"]
sns.countplot(x="spam_target", data=df, palette=palette, ax=axes[0])
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels([f"Not Spam\n({not_spam_total} calls)", f"Spam\n({spam_total} calls)"])
axes[0].set_title("Primary Distribution: Spam vs Not Spam", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Derived Classification Target (spam_target)")
axes[0].set_ylabel("Call Count")

for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height())} ({p.get_height()/len(df)*100:.1f}%)",
                     (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha="center", va="center", color="white", fontweight="bold", fontsize=11)

# 2. Donut Chart
axes[1].pie([not_spam_total, spam_total], labels=["Not Spam", "Spam"],
            autopct="%1.1f%%", colors=palette, startangle=140,
            wedgeprops=dict(width=0.45, edgecolor="w", linewidth=2))
axes[1].set_title("Class Ratio: Spam vs Not Spam", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()


## 5. Secondary Supporting Analysis: Spam Composition
Because the dataset contains the `fraud` column, we provide this secondary intelligence:
Among all Spam records (`spam_target = 1`):
- **Spam / Non-Fraud (`1,0`)**: Telemarketing, nuisance promotions, unsolicited offers.
- **Spam / Fraud (`1,1`)**: Malicious impostor fraud, credential harvesting, financial theft.

> **Important**: This is supporting analysis only. It does **NOT** alter the primary binary classification task.


In [ ]:
# Filter to spam calls only
spam_df = df[df["spam_target"] == 1]
spam_comp_counts = spam_df[fraud_col].value_counts()

comp_labels = ["Spam / Non-Fraud (1,0)", "Spam / Fraud (1,1)"]
comp_values = [c_1_0, c_1_1]
comp_pcts = [round(c_1_0 / spam_total * 100, 2), round(c_1_1 / spam_total * 100, 2)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Horizontal bar chart
bars = axes[0].barh(comp_labels, comp_values, color=["#f59e0b", "#b91c1c"])
axes[0].set_title("Spam Composition Breakdown (Supporting Analysis)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Number of Calls")
for bar in bars:
    w = bar.get_width()
    axes[0].text(w - 25, bar.get_y() + bar.get_height()/2, f"{int(w)} calls",
                 va="center", color="white", fontweight="bold")

# Composition pie chart
axes[1].pie(comp_values, labels=comp_labels, autopct="%1.1f%%",
            colors=["#f59e0b", "#b91c1c"], startangle=90,
            wedgeprops=dict(width=0.45, edgecolor="w", linewidth=2))
axes[1].set_title("Spam Type Share (among 400 Spam Calls)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

comp_table = pd.DataFrame({
    "Category": ["Spam / Non-Fraud (1,0)", "Spam / Fraud (1,1)", "Total Spam"],
    "Call Count": [c_1_0, c_1_1, spam_total],
    "Percentage of Spam": [f"{comp_pcts[0]}%", f"{comp_pcts[1]}%", "100.0%"]
})
comp_table


## 6. EDA — Transcript Length & Boxplots


In [ ]:
df["char_length"] = df["text"].astype(str).str.len()
df["word_count"] = df["text"].astype(str).apply(lambda t: len(t.split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(data=df, x="char_length", hue="spam_target", kde=True,
             palette=["#16a34a", "#dc2626"], bins=30, ax=axes[0])
axes[0].set_title("Transcript Character Length by Class", fontweight="bold")
axes[0].legend(["Spam", "Not Spam"])

sns.boxplot(x="spam_target", y="word_count", data=df,
            palette=["#16a34a", "#dc2626"], ax=axes[1])
axes[1].set_xticklabels(["Not Spam", "Spam"])
axes[1].set_title("Word Count Distribution by Class", fontweight="bold")

plt.tight_layout()
plt.show()


## 7. EDA — Word Clouds for Not Spam vs Spam


In [ ]:
spam_words = " ".join(df[df["spam_target"] == 1]["text"])
legit_words = " ".join(df[df["spam_target"] == 0]["text"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].imshow(WordCloud(width=600, height=380, background_color="white", colormap="Reds").generate(spam_words))
axes[0].set_title("Spam Call Vocabulary (High Urgency & Financial Claims)", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(WordCloud(width=600, height=380, background_color="white", colormap="Greens").generate(legit_words))
axes[1].set_title("Not Spam Call Vocabulary (Conversational & Legitimate Inquiries)", fontsize=12, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()


## 8. Text Preprocessing
Standardized NLP cleaning:
- Lowercasing
- Stripping bracketed placeholders (`r"\[.*?\]"`)
- Stripping non-alphabetic characters
- Whitespace normalization
- NLTK English stop words removal
- WordNet lemmatization


In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\[.*?\]", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join(
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in stop_words and len(w) > 2
    )

df["clean_text"] = df["text"].apply(clean_text)
print("Preprocessing complete. Example comparison:")
print("Original:", df["text"].iloc[3][:120])
print("Cleaned: ", df["clean_text"].iloc[3][:120])


## 9. Train / Test Split & Feature Extraction
### Target Leakage Prevention Note:
The `fraud` column is **strictly excluded** from input features. Features are generated purely from the call text (`clean_text`) via TF-IDF vectorization and Chi-Square selection.
- Features: TF-IDF (1-2 ngrams, min_df=2, max_features=3000)
- Selection: Chi-Square `SelectKBest(chi2, k=1500)`
- Target: `spam_target` (0 = Not Spam, 1 = Spam)
- Split: 80% Train / 20% Test (Stratified)


In [ ]:
# Feature vectorization
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=3000, min_df=2)
X_full = tfidf.fit_transform(df["clean_text"])
y = df["spam_target"].values

k = min(1500, X_full.shape[1])
selector = SelectKBest(chi2, k=k)
X_selected = selector.fit_transform(X_full, y)
print(f"Feature matrix shape after SelectKBest(k={k}): {X_selected.shape}")

# Stratified 80/20 train/test split on spam_target
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples:   {X_train.shape[0]} ({np.bincount(y_train)})")
print(f"Test holdout samples: {X_test.shape[0]} ({np.bincount(y_test)})")


## 10. Train & Compare 5 Machine Learning Models
We train five candidate classifiers directly on `spam_target`:
1. **Logistic Regression**
2. **Naive Bayes (MultinomialNB)**
3. **Support Vector Machine (SVC)**
4. **K-Nearest Neighbors (KNN)**
5. **Random Forest (RandomForestClassifier)**


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(C=10, max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(alpha=0.5),
    "Support Vector Machine": SVC(C=1.0, kernel="linear", probability=True, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []
predictions = {}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    predictions[name] = y_pred

    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(X_test)[:, 1]
        roc = float(roc_auc_score(y_test, y_prob))
    else:
        roc = 0.0

    acc = float(accuracy_score(y_test, y_pred))
    prec = float(precision_score(y_test, y_pred, zero_division=0))
    rec = float(recall_score(y_test, y_pred, zero_division=0))
    f1 = float(f1_score(y_test, y_pred, zero_division=0))

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc, 4),
    })

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
results_df.index = results_df.index + 1
results_df


## 11. 2x2 Binary Confusion Matrices
Evaluating True Negatives, False Positives, False Negatives, and True Positives on the holdout test set across all 5 models.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.ravel()

target_names = ["Not Spam", "Spam"]

for i, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=target_names, yticklabels=target_names,
                ax=axes_flat[i], cbar=False)
    axes_flat[i].set_title(f"{name}", fontweight="bold", fontsize=11)
    axes_flat[i].set_ylabel("Actual Ground Truth")
    axes_flat[i].set_xlabel("Predicted Label")

# Hide extra 6th subplot
fig.delaxes(axes_flat[5])

plt.suptitle("Binary Confusion Matrices (2x2) on 160 Holdout Calls", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


## 12. Hyperparameter Tuning
Optimizing regularization parameters for the top-performing model via 5-Fold Stratified `GridSearchCV`.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print(f"Top performing model selected for tuning: {best_model_name}")

param_grid = {
    "C": [0.1, 1.0, 5.0, 10.0, 20.0],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"],
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print(f"Best Hyperparameters: {grid.best_params_}")
print(f"Best 5-Fold Cross-Validation F1-Score: {grid.best_score_:.4f}")
print(f"Final Test Accuracy:  {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"Final Test F1-Score:  {f1_score(y_test, y_pred_tuned):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_tuned, target_names=["Not Spam", "Spam"]))


## 13. Final Spam vs Not Spam Inference Demonstration
Testing sample call transcripts through the trained pipeline.
The predicted verdict is strictly **SPAM** or **NOT SPAM**.


In [ ]:
sample_calls = [
    "[Greetings], this is the bank fraud division. We have suspended your account due to unauthorized transactions. Please confirm your OTP and credit card PIN immediately to prevent permanent account closure.",
    "[Greetings], this is Sarah from Dr. Miller's dental clinic. Just calling to remind you about your routine cleaning appointment tomorrow at 2 PM. Please let us know if you need to reschedule."
]

for idx, call in enumerate(sample_calls, 1):
    cleaned = clean_text(call)
    vec = selector.transform(tfidf.transform([cleaned]))
    pred = int(best_model.predict(vec)[0])
    prob = best_model.predict_proba(vec)[0]

    verdict = "SPAM" if pred == 1 else "NOT SPAM"
    confidence = prob[pred] * 100

    print(f"\nTest Call #{idx}:")
    print(f"  Transcript: \"{call[:110]}...\"")
    print(f"  Verdict:    {verdict} ({confidence:.1f}% confidence)")
